In [132]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [133]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())

torch.manual_seed(1234)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

True
1
NVIDIA GeForce GTX 1650


'cuda'

In [134]:
# Hyperparameters
train_split = 9
val_split = 1
eval_iters = 200

train_split = train_split/(train_split + val_split)
val_split = val_split/(train_split + val_split)

In [135]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f'{" ".join(chars)}')
print(vocab_size)


   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
65


In [136]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

encode('aads')
decode(encode('aads'))

[39, 39, 42, 57]

'aads'

In [137]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(len(data) * train_split)
train = data[:n]
val = data[n:]

data.shape
train.shape
val.shape

train.shape[0] + val.shape[0]

torch.Size([1115394])

torch.Size([1003854])

torch.Size([111540])

1115394

In [138]:
def get_batch(data, batch_size=4, context_size=8):
  ix = torch.randint(len(data)-context_size, (batch_size,))
  x = torch.stack([data[i:i+context_size] for i in ix])
  y = torch.stack([data[i+1:i+context_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

x, y = get_batch(data)
x.shape
y.shape
x
y


torch.Size([4, 8])

torch.Size([4, 8])

tensor([[ 0, 31, 47, 52, 41, 43,  1, 39],
        [52, 39, 51, 43,  1, 39, 52, 42],
        [63, 53, 59,  1, 40, 43,  1, 49],
        [53, 42,  5, 57,  1, 52, 39, 51]], device='cuda:0')

tensor([[31, 47, 52, 41, 43,  1, 39, 56],
        [39, 51, 43,  1, 39, 52, 42,  1],
        [53, 59,  1, 40, 43,  1, 49, 47],
        [42,  5, 57,  1, 52, 39, 51, 43]], device='cuda:0')

In [145]:
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
  """ A single head of the self attention model"""
  def __init__(self, context_size=8, n_embd=32, head_size=64):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))
    
  def forward(self, x):
    B, T, C = x.shape
    k = self.key(x)
    q = self.query(x)
    # compute attention sccores 'affinities'
    weight = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
    weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    weight = F.softmax(weight, dim=-1)
    # perform the weighted aggregation of values
    v = self.value(x)
    
    out  = weight @ v
    
    return out

  

class BigramLM(nn.Module):
  def __init__(self, vocab_size, embed_dim=32, context_size=8):
    super().__init__()
    self.vocab_size = vocab_size
    self.embed_dim = embed_dim
    self.context_size = context_size
    
    self.token_embedding_table = nn.Embedding(vocab_size, embed_dim)
    self.position_embedding_table = nn.Embedding(context_size, embed_dim)
    self.self_attention_head = Head(context_size, embed_dim, embed_dim)
    self.lm_head = nn.Linear(embed_dim, vocab_size)
  
  def forward(self, idx, targets=None):
    B, T = idx.shape
    
    token_embedding = self.token_embedding_table(idx) # (B, T, C)
    positional_embedding = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
    x = token_embedding + positional_embedding
    x = self.self_attention_head(x) # (B, T, C)
    logits = self.lm_head(x) # (B, T, vocab_size)
    
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_transformed = logits.reshape(B*T, C)
      targets_transformed = targets.reshape(B*T)

      loss = F.cross_entropy(logits_transformed, targets_transformed)
    
    return logits, loss
  
  def generate(self, idx, max_new_tokens=50):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
      idx_continued = idx[:, -self.context_size:]
      logits, loss = self(idx_continued)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    
    return idx

@torch.no_grad()
def estimate_loss(model):
  out = {}
  model.eval()
  
  for split in ['train', 'val']:
    dataset = train if split == 'train' else val
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(data=dataset)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()

  model.train()
  return out


model = BigramLM(vocab_size=vocab_size)

model.to(device)

BigramLM(
  (token_embedding_table): Embedding(65, 32)
  (position_embedding_table): Embedding(8, 32)
  (self_attention_head): Head(
    (key): Linear(in_features=32, out_features=32, bias=False)
    (query): Linear(in_features=32, out_features=32, bias=False)
    (value): Linear(in_features=32, out_features=32, bias=False)
  )
  (lm_head): Linear(in_features=32, out_features=65, bias=True)
)

In [146]:
learning_rate = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

max_iters = 10000
eval_interval = 100

for iter in range(max_iters):
  if iter % eval_interval == 0:
    losses = estimate_loss(model)
    if isinstance(losses, dict):
      print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']: .4f}")
    else:
      print(f"No validation set, so no validation loss.")
  
  xb, yb = get_batch(data=data)
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

step 0: train loss 4.2795, val loss  4.2832
step 100: train loss 3.3405, val loss  3.3737
step 200: train loss 3.2034, val loss  3.2291
step 300: train loss 3.1438, val loss  3.1567
step 400: train loss 3.0636, val loss  3.0963
step 500: train loss 3.0258, val loss  3.0479
step 600: train loss 2.9701, val loss  3.0026
step 700: train loss 2.9488, val loss  2.9645
step 800: train loss 2.9013, val loss  2.9228
step 900: train loss 2.8359, val loss  2.8590
step 1000: train loss 2.8019, val loss  2.7860
step 1100: train loss 2.7432, val loss  2.7591
step 1200: train loss 2.7075, val loss  2.7171
step 1300: train loss 2.7085, val loss  2.6973
step 1400: train loss 2.6697, val loss  2.6423
step 1500: train loss 2.6258, val loss  2.6290
step 1600: train loss 2.6172, val loss  2.6490
step 1700: train loss 2.6205, val loss  2.6218
step 1800: train loss 2.6195, val loss  2.6002
step 1900: train loss 2.6088, val loss  2.5750
step 2000: train loss 2.6072, val loss  2.5740
step 2100: train loss 2.5

In [178]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
raw_response = model.generate(context, max_new_tokens=50)[0].tolist()
response = decode(raw_response)

print(f"Response: {response}")

Response: 
LOT: y.


Gake.

OLUCe wheesen haditot.

KASHARDUK


In [ ]:
# self-attention

B, T, C = 4, 8, 32

x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)

weight = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) = (B, T, T) 

tril = torch.tril(torch.ones(T, T))
# weight = torch.zeros((T, T))
weight = weight.masked_fill(tril == 0, float("-inf"))
weight = F.softmax(weight, dim=1)

out = weight @ x

out.shape
weight[0]

torch.Size([4, 8, 32])

tensor([[0.0264, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0037, 0.3298, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0998, 0.0060, 0.0087, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0543, 0.0348, 0.2647, 0.0211, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1870, 0.0264, 0.1098, 0.1585, 0.3579, 0.0000, 0.0000, 0.0000],
        [0.0083, 0.4036, 0.4899, 0.0477, 0.2927, 0.6393, 0.0000, 0.0000],
        [0.5423, 0.0824, 0.0296, 0.6344, 0.0687, 0.0220, 0.4280, 0.0000],
        [0.0783, 0.1170, 0.0973, 0.1383, 0.2807, 0.3388, 0.5720, 1.0000]],
       grad_fn=<SelectBackward0>)

In [ ]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

weight = q @ k.transpose(-2, -1) * (head_size ** -0.5) # (B, T, 16) @ (B, 16, T) = (T, T)

k.var()
q.var()
weight.var()

tensor(0.9735)

tensor(1.0645)

tensor(0.9409)